In [1]:
import os
import openai
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available
## os.environ directory (env_variablea)
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b",
                    groq_api_key=groq_api_key)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1185fd6a0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1185fe3c0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([
    HumanMessage(content="Hello, I'm an  AI Engineer")
])

AIMessage(content='Hello! 👋 It’s great to meet another AI engineer. How can I assist you today? Are you working on a specific project, looking for resources, or just curious about the latest developments in AI? Let me know, and we can dive right in!', additional_kwargs={'reasoning_content': 'We need to respond to user who says "Hello, I\'m an AI Engineer". We should greet, ask how to help. Possibly ask about what they\'re working on. Also be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 78, 'total_tokens': 178, 'completion_time': 0.137649434, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.003676092, 'prompt_tokens_details': None, 'queue_time': 0.281890232, 'total_time': 0.141325526}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_37c9245f64', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a067a5-9a4a-7c61-ae6d-539b2df7dff7-0', tool_calls

In [4]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hello, I'm Maharshi preparing for AI Engineer"),
    AIMessage(content="Hello! 👋 It’s great to meet an AI Engineer. How can I assist you today?"),
    HumanMessage(content="What is my name and what do I do?")
])

AIMessage(content='You’re **Maharshi**, and you’re preparing to become an **AI Engineer**. 🚀\n\nIn this role you’ll likely:\n\n- Design and build machine‑learning models\n- Work with data pipelines and model deployment\n- Collaborate with cross‑functional teams (data scientists, software engineers, product managers)\n- Stay current with the latest AI research and best practices\n\nLet me know if you’d like help with study plans, interview prep, project ideas, or anything else!', additional_kwargs={'reasoning_content': 'We need to respond: user says "Hello, I\'m Maharshi preparing for AI Engineer". They ask "What is my name and what do I do?" So answer: name Maharshi, preparing for AI Engineer role. Provide brief.'}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 120, 'total_tokens': 276, 'completion_time': 0.190716372, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.006886407, 'prompt_tokens_details': None, 'queue_time': 0.31967593

### Message History

**We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in a some dataStore. Future interaction will then load those messages and pass them into the chain as part of the input.**

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

/var/folders/0b/_fz3pln11xz8q6vfw486xw4r0000gn/T/ipykernel_60436/792372038.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config = {'configurable':{'session_id':'chat1'}}

In [7]:
response = with_message_history.invoke(
    [HumanMessage(content="Hello, I'm Maharshi an AI Engineer")],
    config=config
)

response.content

'Hello Maharshi! 👋 It’s great to meet an AI engineer. How can I assist you today? Whether you’re working on a project, exploring new techniques, or just curious about something, feel free to let me know!'

In [8]:
## Change the config
config1 = {'configurable':{'session_id':'chat2'}}

response = with_message_history.invoke(
    [HumanMessage(content="Hello, What's my name")],
    config=config1
)

response.content

'I’m not sure what your name is—could you let me know?'

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="Hello, My name is John")],
    config=config1
)

response.content

'Nice to meet you, John! How can I help you today?'

In [10]:
response = with_message_history.invoke(
    [HumanMessage(content="Hello, What's my name")],
    config=config1
)

response.content

'Your name is John.'

### Prompt Template

**A prompt template is a reusable framework with dynamic placeholders that get filled with specific variables at runtime to instruct large language models consistentl**

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', "you're a helpful assistant, answer all the questions to the best of your abilities."),
        MessagesPlaceholder(variable_name='messages')
    ]
)

chain = prompt | model

In [12]:
chain.invoke({'messages':[HumanMessage(content="My name is Alice")]}).content

'Nice to meet you, Alice! How can I help you today?'

In [13]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
config2 = {'configurable':{'session_id':'chat3'}}

response = with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Bob")],
    config=config2
)

response

AIMessage(content='Hi Bob! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hello, My name is Bob". We should respond politely. Probably greet them.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 96, 'total_tokens': 137, 'completion_time': 0.043844702, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.004630208, 'prompt_tokens_details': None, 'queue_time': 0.100314592, 'total_time': 0.04847491}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a067a5-aaff-7bc1-8140-3334db2754fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 41, 'total_tokens': 137, 'output_token_details': {'reasoning': 20}})

In [15]:
## add some more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "you're a helpful assistant, answer all the questions to the best of your abilities in {language}"
         ),
        MessagesPlaceholder(variable_name='messages')
    ]
)

chain = prompt | model

In [16]:
response = chain.invoke({'messages':[HumanMessage(content="My name is Alice")], "language":"hindi"})
response.content

'नमस्ते एलिस! आपका नाम बताने के लिए धन्यवाद। आप कैसे हैं? यदि आपको किसी भी विषय पर मदद चाहिए, तो बताइए।'

In [17]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config4 = {'configurable':{'session_id':'chat4'}}

response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hello, My name is Bob")], 'language':'spanish'},
    config=config4
)

In [19]:
response.content

'¡Hola, Bob! ¿En qué puedo ayudarte hoy?'

### Managing the Conversation History

**Managing the conversation history means storing, organizing, and shortening past chat messages so an AI or application remembers what was said without running out of memory or processing space.** 

* **Why It Matters**
    - AI models have strict limits on how much text they can read at once.
    - Proper management stops the AI from forgetting important facts.
    - It keeps running costs and processing speeds low.

### `trim_messages`

**trim_messages can be used to reduce the size of a chat history to a specified token or message count.**

* In either case, if passing the trimmed chat history back into a chat model directly, the resulting chat history should usually satisfy the following properties:

    - The resulting chat history should be valid. Most chat models expect that chat history starts with either 
    (1) a HumanMessage or (2) a SystemMessage followed by a HumanMessage. To achieve this, set `start_on='human'`. In addition, generally a ToolMessage can only appear after an AIMessage that involved a tool call.
    - It includes recent messages and drops old messages in the chat history. To achieve this set the `strategy='last'`.
    - Usually, the new chat history should include the SystemMessage if it was present in the original chat history since the SystemMessage includes special instructions to the chat model. The SystemMessage is almost always the first message in the history if present. To achieve this set the `include_system=True`.

In [20]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=45,
    strategy='last',
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on='human'
)

messages = [
    SystemMessage(content="you're a Good assistant"),
    HumanMessage(content="hi I'm Bob"),
    AIMessage(content="hi"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!")
]

trimmer.invoke(messages)

/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content="you're a Good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [21]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What icecream I like")],
    "language":"english"
    }
)

AIMessage(content='I’m not sure—could you tell me a bit about your favorite flavors or what you’re in the mood for? That’ll help me guess the best ice‑cream for you!', additional_kwargs={'reasoning_content': 'The user asks: "What icecream I like". We need to interpret: They want to know which ice cream they like. There\'s no context. We cannot know. We can ask clarifying question. Provide answer: We can\'t know, ask preferences.We should politely ask for more info.'}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 153, 'total_tokens': 266, 'completion_time': 0.129297405, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.009041371, 'prompt_tokens_details': None, 'queue_time': 0.305699873, 'total_time': 0.138338776}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_92d51d08e5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a067a5-bbcb-7750-9da4-3209436e2de3

In [23]:
response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What math problem did we solve")],
    "language":"english"
    }
)

response.content

'We solved the simple arithmetic problem \\(2 + 2\\), which equals \\(4\\).'

In [25]:
### Lets wrap in message history
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='messages'
)
config = {'configurable':{'session_id':'chat5'}}

In [29]:
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hello, What's my name")],
     'language':'English'},
    config=config
)

response.content

'I’m not sure what your name is—could you tell me?'